# Feature Engineering for Bank Health Classification

The goal of this notebook is to build a table of features that can be used to predict
`bank_condition` (Healthy, Stressed, or Critical), and save that table to
`processed/bank_health_features.csv`.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

## 1. Load the cleaned files

We start from the panel (one row per bank and scenario), and bring in two extra
things: a few bank details that are not already in the panel, and the average risk of
each sector from the loan-level data.

In [2]:
panel = pd.read_csv('../processed/bank_stress_simulated_panel_clean.csv')
bank = pd.read_csv('../processed/bank_profiles_clean.csv')
loan = pd.read_csv('../processed/loan_portfolio_clean.csv')

print('panel:', panel.shape)
print('bank:', bank.shape)
print('loan:', loan.shape)

panel: (20000, 24)
bank: (40, 20)
loan: (3000, 7)


## 2. Bring in the bank details that are missing from the panel

In [3]:
sector_cols = [c for c in bank.columns if c.startswith('sector_wt_')]
extra_bank_cols = ['bank_id', 'deposit_base_usd', 'bank_risk_factor'] + sector_cols

df = panel.merge(bank[extra_bank_cols], on='bank_id', how='left')
print('shape after merge:', df.shape)
df[['bank_id', 'deposit_base_usd', 'bank_risk_factor'] + sector_cols[:2]].head()

shape after merge: (20000, 36)


,bank_id,deposit_base_usd,bank_risk_factor,sector_wt_Technology,sector_wt_Healthcare
0,BANK_001,26039482274,0.079335,0.0378,0.2963
1,BANK_002,17464456027,-0.007972,0.0901,0.0461
2,BANK_003,38665433947,0.017928,0.0576,0.0618
3,BANK_004,17667373015,-0.103565,0.1544,0.0543
4,BANK_005,2360761783,-0.011864,0.0981,0.0898


The panel already has `size_tier`, `sector_concentration`, and the
baseline ratios for each bank, but it does not carry `deposit_base_usd`,
`bank_risk_factor`, or the ten sector weight columns. Those live in
`bank_profiles.csv`, so we join them in using `bank_id`.

## 3. Work out how risky each sector's loans are, on average

In [4]:
sector_avg_pd = loan.groupby('sector')['pd_annual'].mean()
print('Average annual default probability by sector:')
print(sector_avg_pd.sort_values(ascending=False))

Average annual default probability by sector:
sector
Retail         0.019054
Energy         0.016431
Financials     0.010850
Healthcare     0.008070
Utilities      0.007092
Telecom       -2.780372
Consumer      -2.966828
Real_Estate   -3.682074
Industrials   -3.784730
Technology    -5.746476
Name: pd_annual, dtype: float64


This is the same kind of pattern seen in the loan portfolio EDA
notebook: sectors like energy and real estate carry a higher default probability on
average than sectors like utilities and consumer. Knowing this lets us score each bank
by how risky its overall loan book is, based on which sectors it lends to, without
needing to know anything about a specific shock yet.

## 4. Feature: how risky is each bank's loan book (sector risk score)

In [5]:
def sector_risk_score(row):
    score = 0.0
    for sector, avg_pd in sector_avg_pd.items():
        weight_col = f'sector_wt_{sector}'
        if weight_col in row:
            score += row[weight_col] * avg_pd
    return score

df['sector_risk_score'] = df.apply(sector_risk_score, axis=1)
df[['bank_id', 'sector_risk_score']].drop_duplicates().sort_values('sector_risk_score', ascending=False).head()

,bank_id,sector_risk_score
21,BANK_022,-0.937677
36,BANK_037,-1.038980
35,BANK_036,-1.124368
23,BANK_024,-1.266679
27,BANK_028,-1.366962


For each bank, we multiply how much of its lending sits in a sector
by how risky that sector is on average, then add those up. A bank with a lot of
lending in energy and real estate ends up with a higher score than a bank spread
across safer sectors like utilities and consumer. This single number can provide bank's
lending risk that model can use

## 5. Feature: how concentrated is a bank's lending, as a number

In [6]:
df['top_sector_weight'] = df[[f'sector_wt_{s}' for s in sector_avg_pd.index]].max(axis=1)
df['concentration_flag'] = (df['sector_concentration'] == 'concentrated').astype(int)

df[['bank_id', 'top_sector_weight', 'sector_concentration', 'concentration_flag']].drop_duplicates().head()

,bank_id,top_sector_weight,sector_concentration,concentration_flag
0,BANK_001,0.2963,concentrated,1
1,BANK_002,0.1507,diversified,0
2,BANK_003,0.2311,diversified,0
3,BANK_004,0.2009,diversified,0
4,BANK_005,0.2449,diversified,0


`sector_concentration` already tells us diversified versus
concentrated as a label, and `concentration_flag` turns that into a plain 1 or 0 so a
model can use it directly. `top_sector_weight` goes one step further and measures *how*
concentrated a bank is, by taking its single largest sector weight. Two "concentrated"
banks are not necessarily equally risky, one might have 40 percent of its loans in one
sector, another might have 70 percent, and this feature tells them apart.

## 6. Feature: how much room a bank has before it breaks a limit

In [7]:
MIN_CAR_PCT = 11.5
MIN_LIQUIDITY_PCT = 100.0

df['car_buffer'] = df['baseline_car_pct'] - MIN_CAR_PCT
df['liquidity_buffer'] = df['baseline_liquidity_ratio_pct'] - MIN_LIQUIDITY_PCT

df[['bank_id', 'baseline_car_pct', 'car_buffer', 'baseline_liquidity_ratio_pct', 'liquidity_buffer']].drop_duplicates().head()

,bank_id,baseline_car_pct,car_buffer,baseline_liquidity_ratio_pct,liquidity_buffer
0,BANK_001,13.792956,2.292956,131.724447,31.724447
1,BANK_002,14.697693,3.197693,118.030290,18.030290
2,BANK_003,16.383202,4.883202,115.212887,15.212887
3,BANK_004,15.418764,3.918764,123.140971,23.140971
4,BANK_005,13.810658,2.310658,151.581825,51.581825


A bank that starts at a capital ratio of 12 percent is much closer
to trouble than one starting at 18 percent, even though both are technically above the
line. Subtracting a minimum threshold from the baseline ratio turns "where does this
bank start" into "how much room does this bank have before it breaks a limit," which is
a more direct signal of how much a shock can push it over. The two threshold numbers
above are set to common minimums used in this kind of stress test. 

## 7. Feature: balance sheet shape

In [8]:
df['loan_to_asset_ratio'] = df['total_loans_usd'] / df['total_assets_usd']
df['deposit_to_asset_ratio'] = df['deposit_base_usd'] / df['total_assets_usd']

df[['bank_id', 'loan_to_asset_ratio', 'deposit_to_asset_ratio']].drop_duplicates().head()

,bank_id,loan_to_asset_ratio,deposit_to_asset_ratio
0,BANK_001,0.691555,0.803000
1,BANK_002,0.624242,0.789264
2,BANK_003,0.564900,0.775462
3,BANK_004,0.633978,0.654322
4,BANK_005,0.664281,0.848475


The raw dollar figures mostly just tell us how big a bank is,
which `size_tier` already captures. Turning them into ratios tells us how much of the balance sheet is in loans (more loans usually means more defaults), and how much of it is funded by deposits rather than other,
less stable sources of funding.

## 8. Feature: turn scenario severity into a number

In [9]:
SEVERITY_ORDER = {'baseline': 0, 'mild': 1, 'moderate': 2, 'adverse': 3, 'severe': 4}
df['severity_score'] = df['scenario_severity'].map(SEVERITY_ORDER)

df[['scenario_id', 'scenario_severity', 'severity_score']].drop_duplicates().sort_values('severity_score').head(10)

,scenario_id,scenario_severity,severity_score
240,SCN_0007,baseline,0
17880,SCN_0448,baseline,0
17960,SCN_0450,baseline,0
16920,SCN_0424,baseline,0
2960,SCN_0075,baseline,0
4680,SCN_0118,baseline,0
14920,SCN_0374,baseline,0
14520,SCN_0364,baseline,0
4520,SCN_0114,baseline,0
4480,SCN_0113,baseline,0


`scenario_severity` is a set of words, but the words have a real
order to them, baseline is milder than mild, which is milder than moderate, and so on.
Turning that order into the numbers 0 to 4 

## 9. Feature: one combined shock size, instead of six separate numbers

In [10]:
shock_cols = ['gdp_shock_pp', 'unemp_shock_pp', 'rate_shock_pp',
              'credit_spread_bps', 'inflation_shock_pp', 'fx_devaluation_pct']

scaled = pd.DataFrame(index=df.index)
for col in shock_cols:
    magnitude = df[col].abs()
    scaled[col] = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min())

df['shock_severity_score'] = scaled.mean(axis=1)

df[['scenario_id'] + shock_cols + ['shock_severity_score']].drop_duplicates().sort_values('shock_severity_score', ascending=False).head()

,scenario_id,gdp_shock_pp,unemp_shock_pp,rate_shock_pp,credit_spread_bps,inflation_shock_pp,fx_devaluation_pct,shock_severity_score
2720,SCN_0069,-11.08,9.67,5.71,458.3,14.15,42.61,0.893088
40,SCN_0002,-9.76,9.41,5.06,460.1,18.19,40.96,0.880419
2360,SCN_0060,-11.52,9.99,5.32,523.0,12.08,34.13,0.862743
2760,SCN_0070,-12.26,9.98,4.96,499.0,13.05,34.09,0.862008
10880,SCN_0273,-10.55,8.58,5.38,461.1,16.16,38.25,0.859938


 The six shock columns are on very different scales.
`credit_spread_bps` moves in the hundreds, while `inflation_shock_pp` moves in single
digits, so squeezing each
shock column down to a 0 to 1 scale. Once all six are on the same 0 to 1
footing, we average them into one number that stands in for "how big is this shock,
overall.

## 10. Feature: does a shock hit a concentrated bank harder

In [11]:
df['concentration_x_severity'] = df['concentration_flag'] * df['severity_score']
df['risk_x_severity'] = df['bank_risk_factor'] * df['severity_score']

df[['bank_id', 'scenario_id', 'concentration_flag', 'severity_score',
    'concentration_x_severity', 'bank_risk_factor', 'risk_x_severity']].head()

,bank_id,scenario_id,concentration_flag,severity_score,concentration_x_severity,bank_risk_factor,risk_x_severity
0,BANK_001,SCN_0001,1,3,3,0.079335,0.238004
1,BANK_002,SCN_0001,0,3,0,-0.007972,-0.023915
2,BANK_003,SCN_0001,0,3,0,0.017928,0.053783
3,BANK_004,SCN_0001,0,3,0,-0.103565,-0.310696
4,BANK_005,SCN_0001,0,3,0,-0.011864,-0.035591


A concentrated bank and a severe scenario might each look only
moderately risky on their own, but together they could be much worse, becausae to a concentrated
bank a shock hits hard. Multiplying
the two together gives the model a feature that captures this combined effect directly,

We do the same thing for `bank_risk_factor`, since a bank that already carries
more idiosyncratic risk should be more sensitive to a severe scenario than a low-risk
bank 

## 11. Turn size_tier into a number, the same way we did for severity

In [12]:
SIZE_ORDER = {'small': 0, 'medium': 1, 'large': 2}
df['size_score'] = df['size_tier'].map(SIZE_ORDER)

df[['bank_id', 'size_tier', 'size_score']].drop_duplicates().head()

,bank_id,size_tier,size_score
0,BANK_001,medium,1
1,BANK_002,medium,1
2,BANK_003,large,2
3,BANK_004,medium,1
4,BANK_005,small,0


 Same idea as severity: small, medium, and large, we turn them into 0, 1, 2

## 12. Turn the target label into a number too

In [13]:
CONDITION_ORDER = {'Healthy': 0, 'Stressed': 1, 'Critical': 2}
df['bank_condition_code'] = df['bank_condition'].map(CONDITION_ORDER)

df['bank_condition_code'].value_counts().sort_index()

bank_condition_code
0    10291
1     6557
2     3152
Name: count, dtype: int64

We keep the `bank_condition` column too, so the labels are still
easy to read, and add `bank_condition_code` next to it for anything that needs a
number. 

0 for healthy
1 for stressed
2 for critical

## 13. Put together the final feature table

In [14]:
feature_cols = [
    # identifiers, kept for reference, not meant to be fed into a model as-is
    'bank_id', 'scenario_id',

    # bank starting position
    'size_score', 'concentration_flag', 'top_sector_weight', 'sector_risk_score',
    'bank_risk_factor', 'loan_to_asset_ratio', 'deposit_to_asset_ratio',
    'car_buffer', 'liquidity_buffer', 'baseline_roa_pct',

    # the shock itself
    'severity_score', 'shock_severity_score',

    # how the bank and the shock interact
    'concentration_x_severity', 'risk_x_severity',

    # target
    'bank_condition', 'bank_condition_code',
]

features = df[feature_cols].copy()
print('final shape:', features.shape)
features.head()

final shape: (20000, 18)


,bank_id,scenario_id,size_score,concentration_flag,top_sector_weight,sector_risk_score,bank_risk_factor,loan_to_asset_ratio,deposit_to_asset_ratio,car_buffer,liquidity_buffer,baseline_roa_pct,severity_score,shock_severity_score,concentration_x_severity,risk_x_severity,bank_condition,bank_condition_code
0,BANK_001,SCN_0001,1,1,0.2963,-1.401152,0.079335,0.691555,0.803000,2.292956,31.724447,1.584491,3,0.515856,3,0.238004,Healthy,0
1,BANK_002,SCN_0001,1,0,0.1507,-1.852125,-0.007972,0.624242,0.789264,3.197693,18.030290,1.130681,3,0.515856,0,-0.023915,Healthy,0
2,BANK_003,SCN_0001,2,0,0.2311,-2.031593,0.017928,0.564900,0.775462,4.883202,15.212887,0.890601,3,0.515856,0,0.053783,Healthy,0
3,BANK_004,SCN_0001,1,0,0.2009,-2.287426,-0.103565,0.633978,0.654322,3.918764,23.140971,1.236546,3,0.515856,0,-0.310696,Stressed,1
4,BANK_005,SCN_0001,0,0,0.2449,-2.319254,-0.011864,0.664281,0.848475,2.310658,51.581825,1.363997,3,0.515856,0,-0.035591,Healthy,0


## 14. Check for missing values before saving

In [15]:
missing = features.isna().sum()
print(missing[missing > 0])
print()
print('Total rows:', len(features))
print('Rows with at least one missing feature:', features.isna().any(axis=1).sum())

Series([], dtype: int64)

Total rows: 20000
Rows with at least one missing feature: 0


**IMPORTANT:** since the cleaning
notebook turned some MISSING readings into `NaN`. I am leaving them as `NaN` in this file too, since filling
them in is a modelling choice. Whoever trains
a model on this file can decide whether to drop those rows or fill them in.

## 15. Save the final feature table

In [16]:
features.to_csv('../processed/bank_health_features.csv', index=False)
print('Saved processed/bank_health_features.csv')
print('Shape:', features.shape)

Saved processed/bank_health_features.csv
Shape: (20000, 18)


## Summary

The final file, `processed/bank_health_features.csv`, has one row per bank and
scenario

- **About the bank:** its size, how concentrated its lending is, how risky its loan
  book is on average, how much capital and liquidity buffer it starts with, and its
  own risk factor.
- **About the shock:** how severe the scenario is, both as a single ordered score and
  as one combined number built from all six shock variables.
- **How the two combine:** whether a concentrated or already-risky bank is facing a
  more severe scenario.
- **The target:** `bank_condition`, kept as both the original label and a number.

Left out on purpose: `car_after_pct`, `liquidity_after_pct`, `roa_after_pct`,
`projected_npl_ratio_pct`, `stressed_el_rate_pct`, `incremental_credit_loss_usd`, and
`weighted_pd_multiplier`, since all of these are results of the stress simulation
itself, not information available befoer time.